In [1]:
from experiments.bench import harness, plots
import pandas as pd
result = harness.load_result("exp3_algorithm")
print(result["environment"])

{'python': '3.12.13', 'platform': 'macOS-26.5.2-arm64-arm-64bit', 'machine': 'arm64', 'cpu_count': 10, 'cpu_model': 'Apple M4', 'ram': '16.0 GiB', 'cbc': 'Version: 2.10.3', 'neo4j_server': 'Neo4j Kernel 5.26.28 (community)', 'packages': {'numpy': '2.5.1', 'scipy': '1.18.0', 'pulp': '3.3.2', 'neo4j': '6.2.0', 'tiktoken': '0.13.0', 'openai': '2.52.0', 'pandas': '3.0.5'}}


In [2]:
rows_df = pd.DataFrame(result["data"]["rows"])
display(rows_df[["algorithm", "n_people", "pair_cap", "solve_ms",
                  "optimization_ratio", "fulfillment", "unfilled"]]
        .sort_values(["n_people", "algorithm", "pair_cap"]).round(3))

violations_df = pd.DataFrame(result["data"]["violations"])
display(violations_df[["algorithm", "n_people", "pair_cap", "budget_violations"]]
        .sort_values(["n_people", "algorithm", "pair_cap"]))

print("failures (미측정 스케일):",
      [(f["n_people"], f["status"]) for f in result["data"]["failures"]])

,algorithm,n_people,pair_cap,solve_ms,optimization_ratio,fulfillment,unfilled
0,greedy,50,NaN,0.226,0.722,0.456,1
1,milp,50,1000.0,751.438,0.917,0.480,0
2,milp,50,5000.0,705.676,0.917,0.480,0
3,greedy,100,NaN,0.663,0.707,0.461,0
4,milp,100,1000.0,7090.195,0.928,0.544,0
5,milp,100,5000.0,6994.295,0.928,0.544,0
6,greedy,200,NaN,2.422,0.673,0.478,3
7,milp,200,1000.0,23321.143,0.885,0.594,0
8,milp,200,5000.0,44700.059,0.883,0.587,0


,algorithm,n_people,pair_cap,budget_violations
0,greedy,50,NaN,0
1,milp,50,1000.0,0
2,milp,50,5000.0,0
3,greedy,100,NaN,1
4,milp,100,1000.0,0
5,milp,100,5000.0,0
6,greedy,200,NaN,2
7,milp,200,1000.0,0
8,milp,200,5000.0,0


failures (미측정 스케일): [(300, 'skipped'), (500, 'skipped'), (1000, 'skipped')]


In [3]:
plots.exp3_tradeoff(result)   # figures/exp3_tradeoff.png 생성

PosixPath('/Users/honey/Dev/teamweaver/experiments/figures/exp3_tradeoff.png')

In [4]:
from core.config import OPT_RATIO_TARGET
from experiments import report as _report
from IPython.display import Markdown, display

# 최종 리뷰 Important 11(+ Critical 2 반영): 이 셀은 원래 markdown cell로
# 0.9169/0.9278/0.8855, "100인 데모 픽스처", 766.7→6933.5→22628.7 같은 수치를
# 손으로 타이핑해 두고 있었다. 이제 위 코드 셀에서 로드한 result["data"]에서 직접
# 계산해 표시하고, exp3 스윕이 실제로는 커밋된 데모 fixture를 로드하지 않는다는
# 사실도 데이터에서 가져온다.
d = result["data"]
lines = ["## 관찰 요약", ""]
if d.get("data_source"):
    lines.append(f"- **데이터 출처:** {d['data_source']}")

violations = d["violations"]
milp_v = [v for v in violations if v["algorithm"] == "milp"]
greedy_v = sorted((v for v in violations if v["algorithm"] == "greedy"), key=lambda v: v["n_people"])
if milp_v and all(v["budget_violations"] == 0 for v in milp_v):
    lines.append("- MILP는 측정된 모든 규모·모든 `pair_cap`에서 예산 위반 0건이다.")
if greedy_v:
    parts = []
    for v in greedy_v:
        npj, raw = v.get("n_projects"), v["budget_violations"]
        parts.append(f"n={v['n_people']}: {raw}/{npj}건({raw / npj * 100:.0f}%)" if npj
                     else f"n={v['n_people']}: {raw}건")
    lines.append(f"- Greedy의 예산 위반(원시 건수/프로젝트 수 대비 비율): {', '.join(parts)}. "
                 "Greedy는 구조적으로 예산 로직이 없다 — 각 프로젝트의 인력을 점수순으로 다 "
                 "배정한 *뒤에야* 그 프로젝트의 누적 비용을 예산과 비교해 위반 여부만 *기록*할 "
                 "뿐 어떤 후보도 거부하지 않는다(`core/optimize/greedy.py`) — 그래서 위 숫자는 "
                 "저비용의 제약-인지 휴리스틱이 없다는 근거가 아니라, 예산 로직을 아예 생략한 "
                 "베이스라인의 구조적 대가다.")

milp_rows = [r for r in d["rows"] if r["algorithm"] == "milp" and r["pair_cap"] is not None]
if milp_rows:
    min_cap = min(r["pair_cap"] for r in milp_rows)
    base_rows = sorted((r for r in milp_rows if r["pair_cap"] == min_cap), key=lambda r: r["n_people"])
    parts = ", ".join(f"n={r['n_people']}: {r['optimization_ratio']:.4f}" for r in base_rows)
    below = [r for r in base_rows if r["optimization_ratio"] < OPT_RATIO_TARGET]
    below_note = ""
    if below:
        below_ns = ", ".join(f"n={r['n_people']}" for r in below)
        below_note = (f" **{below_ns}에서 {OPT_RATIO_TARGET:.0%} 목표선 아래로 떨어진다** — 이 "
                      "스윕이 기준으로 삼은 n=100은 seed=42로 새로 생성한 데이터셋이지 커밋된 "
                      "데모 fixture가 아니다.")
    lines.append(f"- MILP(pair_cap={min_cap})의 `optimization_ratio`는 {parts}이다.{below_note}")

    fixed = _report._frozen_fixture_ratio()
    n100_row = next((r for r in base_rows if r["n_people"] == 100), None)
    if fixed is not None and n100_row is not None:
        lines.append(f"- 참고로 커밋된 데모 fixture(LLM 모드 리뷰, `README.md`의 E2E 스모크 기록)는 "
                     f"n=100에서 optimization_ratio={fixed:.4f}를 낸다 — 이 스윕의 n=100 값"
                     f"({n100_row['optimization_ratio']:.4f})과는 다른 데이터(인원·프로젝트 수는 "
                     "같지만 리뷰 텍스트가 달라 C 행렬이 다름)에서 나온 수치이므로 섞어 인용하지 "
                     "말 것.")

    sample_params = next((r.get("milp_params") for r in base_rows if r.get("milp_params")), None)
    if sample_params:
        lines.append(f"- `optimization_ratio`는 분자·분모 모두 스킬 항만 잰다"
                     "(`core/optimize/metrics.py`) — 배치 자체는 λ={:.2f}, μ={:.2f}인 스킬+시너지"
                     "-클리크 다중 항 목적함수로 solve된 것이다.".format(
                         sample_params["lam"], sample_params["mu"]))

caps_present = sorted({r["pair_cap"] for r in milp_rows}) if milp_rows else []
if len(caps_present) >= 2:
    lo_cap, hi_cap = caps_present[0], caps_present[-1]
    deltas, mults = [], []
    for n in sorted({r["n_people"] for r in milp_rows}):
        r_lo = next((r for r in milp_rows if r["n_people"] == n and r["pair_cap"] == lo_cap), None)
        r_hi = next((r for r in milp_rows if r["n_people"] == n and r["pair_cap"] == hi_cap), None)
        if r_lo and r_hi:
            deltas.append(abs(r_hi["optimization_ratio"] - r_lo["optimization_ratio"]))
            mults.append(r_hi["solve_ms"] / r_lo["solve_ms"] if r_lo["solve_ms"] else float("nan"))
    if deltas:
        lines.append(f"- `pair_cap`을 {lo_cap}→{hi_cap}로 올려도 최적화율 차이는 최대 "
                     f"{max(deltas):.4f}(잡음 수준)다. 반면 소요 시간 배율은 {min(mults):.2f}×~"
                     f"{max(mults):.2f}× 범위다 — cap을 올리는 것이 품질 이득 없이 시간만 태우는 "
                     "규모가 있다.")

failures = d.get("failures") or []
if failures:
    ns = ", ".join(str(f["n_people"]) for f in sorted(failures, key=lambda x: x["n_people"]))
    lines.append(f"- n≥{min(f['n_people'] for f in failures)}({ns})은 **미측정**이다"
                 f"(`data.failures`). 사유: {failures[0]['reason']}")

display(Markdown("\n".join(lines)))

## 관찰 요약

- **데이터 출처:** 이 스윕의 모든 행은 datasets.build_scale()이 규모마다 새로 생성한 데이터셋을 쓴다 (core.datagen.generator, seed=42 고정, rule-based 파싱의 템플릿 리뷰) — 커밋된 LLM 모드 데모 fixture(fixtures/*.json)를 로드하지 않는다. n=100 지점도 인원·프로젝트 수만 fixture와 같을 뿐 리뷰 텍스트가 달라 C 행렬과 optimization_ratio가 fixture와 다르다 — 프로젝트 개수/인원 수가 같다고 같은 데이터가 아니다.
- MILP는 측정된 모든 규모·모든 `pair_cap`에서 예산 위반 0건이다.
- Greedy의 예산 위반(원시 건수/프로젝트 수 대비 비율): n=50: 0/10건(0%), n=100: 1/20건(5%), n=200: 2/40건(5%). Greedy는 구조적으로 예산 로직이 없다 — 각 프로젝트의 인력을 점수순으로 다 배정한 *뒤에야* 그 프로젝트의 누적 비용을 예산과 비교해 위반 여부만 *기록*할 뿐 어떤 후보도 거부하지 않는다(`core/optimize/greedy.py`) — 그래서 위 숫자는 저비용의 제약-인지 휴리스틱이 없다는 근거가 아니라, 예산 로직을 아예 생략한 베이스라인의 구조적 대가다.
- MILP(pair_cap=1000)의 `optimization_ratio`는 n=50: 0.9169, n=100: 0.9278, n=200: 0.8855이다. **n=200에서 90% 목표선 아래로 떨어진다** — 이 스윕이 기준으로 삼은 n=100은 seed=42로 새로 생성한 데이터셋이지 커밋된 데모 fixture가 아니다.
- 참고로 커밋된 데모 fixture(LLM 모드 리뷰, `README.md`의 E2E 스모크 기록)는 n=100에서 optimization_ratio=0.9300를 낸다 — 이 스윕의 n=100 값(0.9278)과는 다른 데이터(인원·프로젝트 수는 같지만 리뷰 텍스트가 달라 C 행렬이 다름)에서 나온 수치이므로 섞어 인용하지 말 것.
- `optimization_ratio`는 분자·분모 모두 스킬 항만 잰다(`core/optimize/metrics.py`) — 배치 자체는 λ=0.30, μ=0.20인 스킬+시너지-클리크 다중 항 목적함수로 solve된 것이다.
- `pair_cap`을 1000→5000로 올려도 최적화율 차이는 최대 0.0026(잡음 수준)다. 반면 소요 시간 배율은 0.94×~1.92× 범위다 — cap을 올리는 것이 품질 이득 없이 시간만 태우는 규모가 있다.
- n≥300(300, 500, 1000)은 **미측정**이다(`data.failures`). 사유: extrapolated beyond practical session budget: milp cap=1000 solve_ms 766.7 -> 6933.5 -> 22628.7 at n=50/100/200 (9.04x, then 3.26x per doubling); n=300 was attempted twice and killed (CBC healthy, not hung, but not converging inside budget) before this skip list was added